# Investigating the Met's Collection
### What Gets Displayed, What Gets Collected, and What Art Reveals About Its Era

DS3000 Final Project

Group: Anna Bennett, Zara Ceraj 

**1. (20 pts) Expresses the central motivation of the project and explains the (at least) two key questions to be explored. Gives a summary of the data processing pipeline so a technical expert can easily follow along.**

### Central Motivation and Key Questions 
When you walk into a large art museum, it feels like you could spend a week walking through carefully looking at every piece. However, most museums only show a fraction of their collections. For instance, "the Tate shows about 20% of its permanent collection. The Louvre shows 8%, the Guggenheim a lowly 3% and the Berlinische Galerie – a Berlin museum whose mandate is to show, preserve and collect art made in the city – 2% of its holdings" (Bradley, 2015). Physical space is part of the reason, but what ends up on the walls also reflects each institution's curatorial priorities. My questions arise out of these inaccessible collections that many museums have, focusing more specifically on the Metropolitan Museum of Art.

Question 1: What predicts whether a work is on display?

Can features like an artwork's age, size, department, culture of origin, or the artist's gender predict whether it hangs in a gallery or sits in storage?

What gets displayed depends on what was collected in the first place, and collecting has historically been uneven. In 1989, the Guerrilla Girls, an anonymous collective of feminist artists, visited the Met and compared the number of women artists in its modern art galleries with the number of female nudes on display. They found that fewer than 5% of the artists in those galleries were women, while 85% of the nudes were female (Guerrilla Girls, 1989). Decades later, the gap remains: between 2008 and 2018, "works by women constituted just 11 percent of acquisitions and 14 percent of exhibitions at 26 major American museums" (Solly, 2019).

Artistic movements are often defined by new materials, techniques, or philosophies, but there is overlap across very different eras. Oil paint, for example, was used centuries apart by both Renaissance and Impressionist painters. 

Question 2: How accurately can an artwork's time period be predicted from its traits, such as its medium, size, and culture of origin?

### Data Processing Pipeline
-  **Get all object IDs:** A GET request to the Met Collection API's /objects endpoint returns the IDs of every object in the collection (~502,000).
- **Sample:** Because the API returns only one object per request, we draw a random sample of 500 IDs (random.seed(3000) for reproducibility).
- **Fetch each object:** For each sampled ID, we request /objects/{id} with a browser User-Agent header. --> A failed request (e.g., a 403 from rate limiting) is retried up to 3 times with increasing waits (5, 10, 15 s), and a 1 s pause follows each successful request. Objects that still fail are skipped and counted.
- **Extract features:** From each JSON record we keep 13 fields: objectID, title, department, classification, culture, medium, artistDisplayName, artistGender, objectBeginDate, objectEndDate, accessionYear, and GalleryNumber. Height and width (cm) are arsed from the nested measurements list, using the element named `"Overall"; they are left null if missing.
- **Build the DataFrame and clean:** The rows are combined into a pandas DataFrame (500 rows × 15 columns). 
- **Save:** The result is written to met_sample.csv. 

**2. (25 pts) Obtains, cleans, and merges all data sources involved in the project.**

## Hi, note: the data hasn't fully been cleaned yet, theres one date still in the future and some empty data points that idk how we want to handle 

### Data Source
The Metropolitan Museum of Art (the Met) in New York City is one of the largest art museums in the world, with a collection spanning thousands of years. The museum publishes data on over 502,000 artworks through a free public API that requires no registration or key. 

Each artwork's record includes dozens of fields, such as when the work was made, when the museum acquired it, its dimensions, its department and culture of origin, and whether it is currently on view.

[Data Source: The Metropolitan Museum of Art Collection API](https://metmuseum.github.io/)

In [ ]:
import random
import time

import pandas as pd
import requests

In [ ]:
url = "https://collectionapi.metmuseum.org/public/collection/v1/objects"

response = requests.get(url)

all_ids = response.json()["objectIDs"]
print("Total objects in the collection:", len(all_ids))

random.seed(3000)
sample_ids = random.sample(all_ids, 500)

Total objects in the collection: 502886


Note: Due to the size of the collection and since the API only returns one object per request, I took a random sample of 500 IDs. This sample might not accurately reflect the full collection, but the API's rate limits made larger-scale collection impractical.

In [ ]:
rows = []
failed = 0

for object_id in sample_ids:
    # retry up to 3 times, waiting longer each time, since the API blocks fast repeated requests
    # Per syllabus: I used AI to help me write this chunk as I kept getting 403 errors when I tried to run the code 
    # without a user agent and without a retry mechanism.
    response = None
    for attempt in range(3):
        response = requests.get(url + "/" + str(object_id), headers={"User-Agent": "Mozilla/5.0"})
        if response.status_code == 200:
            break
        time.sleep(5 * (attempt + 1))
    if response.status_code != 200:
        failed += 1
        continue
    obj = response.json()

    # find the overall height and width but some have missing data 
    height = None
    width = None
    if obj["measurements"] is not None:
        for m in obj["measurements"]:
            if m["elementName"] == "Overall":
                sizes = m["elementMeasurements"]
                if "Height" in sizes:
                    height = sizes["Height"]
                if "Width" in sizes:
                    width = sizes["Width"]

    row = {
        "objectID": obj["objectID"],
        "title": obj["title"],
        "department": obj["department"],
        "classification": obj["classification"],
        "culture": obj["culture"],
        "medium": obj["medium"],
        "artistDisplayName": obj["artistDisplayName"],
        "artistGender": obj["artistGender"],
        "objectBeginDate": obj["objectBeginDate"],
        "objectEndDate": obj["objectEndDate"],
        "accessionYear": obj["accessionYear"],
        "height_cm": height,
        "width_cm": width,
        "GalleryNumber": obj["GalleryNumber"],
    }
    rows.append(row)

    time.sleep(1) # had to keep extending this... 

print("Failed requests:", failed)

met_df = pd.DataFrame(rows)
met_df.head()


Failed requests: 0


,objectID,title,department,classification,culture,medium,artistDisplayName,artistGender,objectBeginDate,objectEndDate,accessionYear,height_cm,width_cm,GalleryNumber
0,502128,Sek Kei,Musical Instruments,Idiophone-Struck-gong,Japanese,"Stone, wood",,,1801,1900,1889,NaN,NaN,
1,81088,Dress,Costume Institute,,Belgian,leather,Ann Demeulemeester,,1997,1997,1998,NaN,NaN,
2,268387,St. Andrews. The Pends,Photographs,Photographs,,Salted paper print from paper negative,Hill and Adamson,,1843,1847,1937,NaN,NaN,
3,51453,Bird Pendant,Asian Art,Jewelry,"Indonesia (Java, Lumajang, Pasiran)",Bronze,,,-500,300,2001,2.10,3.5,
4,248894,Bronze box mirror,Greek and Roman Art,Bronzes,Greek,Bronze,,,-330,-300,1914,14.81,NaN,158


In [ ]:
# cleaning and simplifying the data
met_df["onView"] = met_df["GalleryNumber"] != ""
met_df["accessionYear"] = pd.to_numeric(met_df["accessionYear"], errors="coerce")

met_df.to_csv("met_sample.csv", index=False)

print("DataFrame shape:", met_df.shape)

print("\nDescription:")
met_df.describe()

DataFrame shape: (500, 15)

Description:


,objectID,objectBeginDate,objectEndDate,accessionYear,height_cm,width_cm
count,500.000000,500.000000,500.000000,493.000000,180.000000,174.000000
mean,413411.926000,1327.874000,1438.996000,1958.217039,33.180069,31.250916
std,259402.602253,1271.148238,1013.657002,35.701671,48.674256,51.481212
min,952.000000,-10025.000000,-4975.000000,1874.000000,0.317501,0.400000
25%,219819.250000,1590.000000,1643.750000,1931.000000,8.890018,9.500000
50%,394838.500000,1800.000000,1855.000000,1963.000000,19.150000,17.790018
75%,635334.750000,1898.250000,1909.250000,1985.000000,32.096266,31.750063
max,977999.000000,2015.000000,2015.000000,2056.000000,331.500000,535.941100


Note: the dates make sense because they used negative dates for BC and positive for AD. 

However, one accession year is listed as 2056, which is likely a data entry error I will remove during further cleaning.

**3. (25 pts) Builds at least two visualizations (graphs/plots) from the data which help to understand or answer the questions of interest. These visualizations will be graded based on how much information they can effectively communicate to readers. Please make sure your visualization are sufficiently distinct from each other.**

**4. (20 pts) Develop at least two functions that will verify fit or other attributes of analysis from a ML model. (You don't need to identify a specific model yet).  Include markup sections which explain why these functions will be useful in analyzing a ML model.**

**5. (10 pts) Discuss potential ethical implications of the data and using ML to analyze it.**

### References

Bradley, K. (2015, January 23). Why museums hide masterpieces away. BBC Culture. [https://www.bbc.com/culture/article/20150123-7-masterpieces-you-cant-see](https://www.bbc.com/culture/article/20150123-7-masterpieces-you-cant-see)

Guerrilla Girls. (1989). Do women have to be naked to get into the Met. Museum? The Metropolitan Museum of Art, New York, NY. [https://www.metmuseum.org/art/collection/search/849438](https://www.metmuseum.org/art/collection/search/849438)

Solly, M. (2019, September 20). Study shows U.S. museums still lag when it comes to acquiring works by women artists. Smithsonian Magazine. [https://www.smithsonianmag.com/smart-news/survey-us-museums-reveals-few-gains-female-artists-past-decade-180973190/](https://www.smithsonianmag.com/smart-news/survey-us-museums-reveals-few-gains-female-artists-past-decade-180973190/)